<a href="https://colab.research.google.com/github/Tonstonte/credit-scoring-xai-project_uniabj/blob/main/GERMAN_CREDIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LOAD AND CLEAN

In [ ]:
"""
01_load_and_clean_german.py
============================
Data loading and preprocessing for the German Credit dataset.
Thesis: Evaluating the Trade-off Between Predictive Accuracy and
        Interpretability in Credit Scoring (XAI Comparative Study)

Dataset
-------
Source  : Kaggle — German Credit Risk (Rohan Paris / UCI origin)
Rows    : 1,000
Features: 10 raw → 15 after encoding
Target  : Risk (good=0, bad=1)
Imbalance: 2.33:1 (700 good, 300 bad) — milder than Give Me Some Credit (14:1)

Key preprocessing decisions
----------------------------
1. NaN as "none" category (savings: 183, checking: 394 missing)
   Rationale: missing account information reflects genuinely unbanked
   borrowers — a distinct credit risk profile, not a data error.
   Default rates confirm this: none≈little for checking accounts (48.5% vs 49.3%).
   Mode imputation would artificially inflate the "little" category.

2. Ordinal encoding for savings, checking, housing
   These are ordered categories — ordinal encoding preserves the gradient
   (more savings → lower risk) that one-hot encoding would lose.

3. One-hot encoding for Sex and Purpose
   No natural ordering; dummy encoding with drop_first to avoid multicollinearity.

4. Binary target: bad=1, good=0

Dependencies
------------
pip install pandas numpy scikit-learn
"""

import pandas as pd
import numpy as np

# ── 1. Load ───────────────────────────────────────────────────────────────────

df = pd.read_csv("german_credit_data.csv", index_col=0)
print(f"Raw shape: {df.shape}")
print(f"Missing: Saving accounts={df['Saving accounts'].isnull().sum()}, "
      f"Checking account={df['Checking account'].isnull().sum()}")

# ── 2. NaN → 'none' category ──────────────────────────────────────────────────
#
# Treating missingness as a distinct category is preferred over imputation
# because:
#   a) The missingness rate is high (39.4% for checking account) — imputing
#      would distort the distribution significantly.
#   b) Domain logic supports it: a borrower with no recorded account likely
#      has no account, not a missing record.
#   c) Default rates for 'none' cluster with 'little', confirming it is
#      a meaningful category rather than random missingness.

df['Saving accounts']  = df['Saving accounts'].fillna('none')
df['Checking account'] = df['Checking account'].fillna('none')

# ── 3. Ordinal encoding ───────────────────────────────────────────────────────

saving_order   = {'none': 0, 'little': 1, 'moderate': 2, 'quite rich': 3, 'rich': 4}
checking_order = {'none': 0, 'little': 1, 'moderate': 2, 'rich': 3}
housing_order  = {'free': 0, 'rent': 1, 'own': 2}

df['Saving accounts']  = df['Saving accounts'].map(saving_order)
df['Checking account'] = df['Checking account'].map(checking_order)
df['Housing']          = df['Housing'].map(housing_order)

# ── 4. One-hot encode Sex and Purpose ────────────────────────────────────────

df = pd.get_dummies(df, columns=['Sex', 'Purpose'], drop_first=True)

# Convert bool columns to int for sklearn compatibility
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

# ── 5. Encode target ──────────────────────────────────────────────────────────

df['Risk'] = (df['Risk'] == 'bad').astype(int)

# ── 6. Validate ───────────────────────────────────────────────────────────────

assert df.isnull().sum().sum() == 0, "Unexpected missing values!"
print(f"\nFinal shape  : {df.shape}")
print(f"Features     : {[c for c in df.columns if c != 'Risk']}")
print(f"Target dist  : good={( df['Risk']==0).sum()}  bad={(df['Risk']==1).sum()}")
print(f"Imbalance    : {(df['Risk']==0).sum()/(df['Risk']==1).sum():.2f}:1")

# ── 7. Save ───────────────────────────────────────────────────────────────────

df.to_csv("german_credit_cleaned.csv")
print("\nSaved → german_credit_cleaned.csv")

# ── 8. EDA summary ───────────────────────────────────────────────────────────

print("\n── Default rates by key feature ────────────────────────────────────────")
for c in ['Checking account', 'Saving accounts', 'Housing']:
    grp = df.groupby(c)['Risk'].mean().round(3)
    print(f"\n{c}:")
    print(grp)

print("\n── Numeric feature means by Risk ───────────────────────────────────────")
print(df.groupby('Risk')[['Age','Credit amount','Duration']].mean().round(1))

print("\n── Pearson correlation with target ─────────────────────────────────────")
num_feats = ['Age','Job','Housing','Saving accounts','Checking account','Credit amount','Duration']
corr = df[num_feats + ['Risk']].corr()['Risk'].drop('Risk').sort_values(key=abs, ascending=False)
print(corr.round(3))

print("\nPreprocessing complete. Proceed to 02_logistic_regression_german.py")

Raw shape: (1000, 10)
Missing: Saving accounts=183, Checking account=394

Final shape  : (1000, 16)
Features     : ['Age', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Sex_male', 'Purpose_car', 'Purpose_domestic appliances', 'Purpose_education', 'Purpose_furniture/equipment', 'Purpose_radio/TV', 'Purpose_repairs', 'Purpose_vacation/others']
Target dist  : good=700  bad=300
Imbalance    : 2.33:1

Saved → german_credit_cleaned.csv

── Default rates by key feature ────────────────────────────────────────

Checking account:
Checking account
0    0.117
1    0.493
2    0.390
3    0.222
Name: Risk, dtype: float64

Saving accounts:
Saving accounts
0    0.175
1    0.360
2    0.330
3    0.175
4    0.125
Name: Risk, dtype: float64

Housing:
Housing
0    0.407
1    0.391
2    0.261
Name: Risk, dtype: float64

── Numeric feature means by Risk ───────────────────────────────────────
       Age  Credit amount  Duration
Risk                               
0   

# LOGISTIC REGRESSIOM

In [ ]:
"""
02_logistic_regression_german.py
=================================
Logistic Regression baseline model for German Credit dataset.
Thesis: Evaluating the Trade-off Between Predictive Accuracy and
        Interpretability in Credit Scoring (XAI Comparative Study)

Results
-------
ROC-AUC : 0.6635  (5-fold CV: 0.6813 ± 0.0332)
PR-AUC  : 0.4837
Bad recall    : 58.3%  (35/60 defaults caught)
Bad precision : 43.8%
Confusion matrix: TN=95  FP=45  FN=25  TP=35

Top coefficients (standardised log-odds):
  Duration (+0.579) — longer loans = higher risk
  Checking account (+0.563) — thinner balance = higher risk
  Education purpose (+0.265) — riskiest loan type
  Age (-0.146) — older borrowers slightly safer
  Sex_male (-0.136) — fairness flag

Dependencies
------------
pip install scikit-learn imbalanced-learn pandas numpy matplotlib
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score,
    average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TEST_SIZE    = 0.2

# ── 1. Load ───────────────────────────────────────────────────────────────────

df = pd.read_csv("german_credit_cleaned.csv", index_col=0)
X  = df.drop("Risk", axis=1)
y  = df["Risk"]
feature_names = list(X.columns)

# ── 2. Split ──────────────────────────────────────────────────────────────────
#
# Stratified split preserves the 30% bad rate in both train and test.
# No customer-level leakage concern here — each row is a unique applicant.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

# ── 3. Scale + SMOTE ─────────────────────────────────────────────────────────
#
# SMOTE oversample minority (bad) class to balance training set.
# Imbalance ratio is 2.33:1 — milder than Give Me Some Credit (14:1).
# SMOTE still helps LR find the decision boundary for the minority class.

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

sm           = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = sm.fit_resample(X_train_sc, y_train)
print(f"After SMOTE — good: {(y_res==0).sum()}  bad: {(y_res==1).sum()}")

# ── 4. Train ──────────────────────────────────────────────────────────────────

lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_res, y_res)

# ── 5. Evaluate ───────────────────────────────────────────────────────────────

y_pred  = lr.predict(X_test_sc)
y_proba = lr.predict_proba(X_test_sc)[:, 1]

roc_auc = roc_auc_score(y_test, y_proba)
pr_auc  = average_precision_score(y_test, y_proba)

print(f"\nROC-AUC : {roc_auc:.4f}")
print(f"PR-AUC  : {pr_auc:.4f}")
print(classification_report(y_test, y_pred, target_names=["Good", "Bad"]))

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

# ── 6. Cross-validation ───────────────────────────────────────────────────────

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(lr, X_train_sc, y_train, cv=cv, scoring="roc_auc")
print(f"\n5-fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# ── 7. Coefficients ───────────────────────────────────────────────────────────

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef"   : lr.coef_[0]
}).sort_values("coef", key=abs, ascending=False).reset_index(drop=True)

print("\nCoefficients (standardised log-odds):")
print(coef_df.to_string(index=False))

# ── 8. Plots ──────────────────────────────────────────────────────────────────

# ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color="#3266ad", lw=2, label=f"LR (AUC={roc_auc:.3f})")
ax.plot([0,1],[0,1], color="gray", lw=1, linestyle="--")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — LR (German Credit)"); ax.legend()
plt.tight_layout(); plt.savefig("roc_lr_german.png", dpi=150); plt.close()

# PR
prec, rec, _ = precision_recall_curve(y_test, y_proba)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(rec, prec, color="#3266ad", lw=2, label=f"LR (AP={pr_auc:.3f})")
ax.axhline(y=y_test.mean(), color="gray", lw=1, linestyle="--", label="No-skill")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("PR Curve — LR (German Credit)"); ax.legend()
plt.tight_layout(); plt.savefig("pr_lr_german.png", dpi=150); plt.close()

# Coefficient plot
fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#d85a30" if c > 0 else "#2a7a3b" for c in coef_df["coef"]]
ax.barh(coef_df["feature"][::-1], coef_df["coef"][::-1], color=colors[::-1])
ax.axvline(0, color="gray", lw=0.8, linestyle="--")
ax.set_xlabel("Standardised coefficient (log-odds)")
ax.set_title("Logistic Regression Coefficients — German Credit")
plt.tight_layout(); plt.savefig("coef_lr_german.png", dpi=150); plt.close()

print("\nSaved -> roc_lr_german.png, pr_lr_german.png, coef_lr_german.png")
print("Proceed to 03_shap_lr_german.py")

After SMOTE — good: 560  bad: 560

ROC-AUC : 0.6635
PR-AUC  : 0.4837
              precision    recall  f1-score   support

        Good       0.79      0.68      0.73       140
         Bad       0.44      0.58      0.50        60

    accuracy                           0.65       200
   macro avg       0.61      0.63      0.62       200
weighted avg       0.69      0.65      0.66       200

TN=95  FP=45  FN=25  TP=35

5-fold CV ROC-AUC: 0.6813 ± 0.0332

Coefficients (standardised log-odds):
                    feature      coef
                   Duration  0.579245
           Checking account  0.563034
          Purpose_education  0.265178
                Purpose_car  0.168906
Purpose_furniture/equipment  0.166590
            Purpose_repairs  0.151104
                        Age -0.146462
                   Sex_male -0.136010
    Purpose_vacation/others -0.121287
                    Housing -0.118604
           Purpose_radio/TV -0.086549
            Saving accounts -0.078910
        

In [ ]:
"""
03_shap_lr_german.py
====================
SHAP LinearExplainer analysis for Logistic Regression on German Credit.
Thesis: Evaluating the Trade-off Between Predictive Accuracy and
        Interpretability in Credit Scoring (XAI Comparative Study)

KEY FAIRNESS FINDING
--------------------
Sex_male SHAP values are constant across all borrowers (LR is linear):
  Male   SHAP = −0.112  (reduces default probability)
  Female SHAP = +0.182  (increases default probability)

Counterfactual gap: same borrower, female → +6.18pp higher default probability.

This constitutes direct algorithmic discrimination under:
  - EU AI Act Article 10 (prohibited grounds include sex)
  - Nigeria Data Protection Act 2023 (fairness principle)
  - General credit regulation principles (protected characteristics)

SHAP makes this legible. A coefficient of −0.136 requires log-odds
understanding; SHAP's +0.182 translates directly to human-readable impact.

Dependencies
------------
pip install scikit-learn imbalanced-learn shap pandas numpy matplotlib
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import shap
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TEST_SIZE    = 0.2
N_BACKGROUND = 500

# ── 1. Rebuild model ──────────────────────────────────────────────────────────

df = pd.read_csv("german_credit_cleaned.csv", index_col=0)
X  = df.drop("Risk", axis=1)
y  = df["Risk"]
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

sm           = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = sm.fit_resample(X_train_sc, y_train)

lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_res, y_res)
y_proba = lr.predict_proba(X_test_sc)[:, 1]
print("Model retrained.")

# ── 2. SHAP LinearExplainer ───────────────────────────────────────────────────

np.random.seed(RANDOM_STATE)
bg_idx     = np.random.choice(len(X_res), N_BACKGROUND, replace=False)
background = X_res[bg_idx]

explainer = shap.LinearExplainer(
    lr, background, feature_perturbation="interventional"
)
sv       = explainer.shap_values(X_test_sc)
base_val = explainer.expected_value
print(f"Base value: {base_val:.4f}")

# ── 3. Global importance ──────────────────────────────────────────────────────

mean_abs = np.abs(sv).mean(axis=0)
imp_df   = pd.DataFrame({
    "feature"      : feature_names,
    "mean_abs_shap": mean_abs
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

print("\n── Global mean |SHAP| ──────────────────────────────────────────────────")
print(imp_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#d85a30" if "Sex" in f else "#3266ad" for f in imp_df["feature"][::-1]]
ax.barh(imp_df["feature"][::-1], imp_df["mean_abs_shap"][::-1], color=colors)
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("SHAP Global Importance — LR (German Credit)\n(red = fairness-sensitive feature)")
plt.tight_layout()
plt.savefig("shap_global_lr_german.png", dpi=150)
plt.close()
print("Saved -> shap_global_lr_german.png")

# ── 4. Fairness deep dive — Sex_male ─────────────────────────────────────────
#
# For a binary feature in a linear model, SHAP values are constant per class.
# This makes the discrimination legible: every female applicant receives
# SHAP = +0.182 for sex, every male receives −0.112, regardless of all
# other features. This is direct, unconditional sex-based discrimination.

sex_idx   = feature_names.index("Sex_male")
male_mask = X_test["Sex_male"].values == 1

print("\n── Sex_male SHAP (fairness analysis) ───────────────────────────────────")
print(f"Rank: {imp_df[imp_df['feature']=='Sex_male'].index[0]+1} of {len(feature_names)}")
print(f"Male   SHAP = {sv[male_mask, sex_idx].mean():.4f}  (n={male_mask.sum()})")
print(f"Female SHAP = {sv[~male_mask, sex_idx].mean():.4f}  (n={(~male_mask).sum()})")
print(f"Male   default rate = {y_test[male_mask].mean():.3f}")
print(f"Female default rate = {y_test[~male_mask].mean():.3f}")

# Counterfactual analysis
X_cf_male   = X_test_sc.copy()
X_cf_female = X_test_sc.copy()
X_cf_male[:, sex_idx]   = X_test_sc[:, sex_idx].max()
X_cf_female[:, sex_idx] = X_test_sc[:, sex_idx].min()

gap = lr.predict_proba(X_cf_female)[:,1].mean() - lr.predict_proba(X_cf_male)[:,1].mean()
print(f"\nCounterfactual gap (same borrower, female vs male): {gap:+.4f} ({gap*100:.2f}pp)")
print("Interpretation: being female adds ~6.2pp to predicted default probability")
print("This constitutes direct algorithmic sex discrimination.")

# ── 5. Beeswarm ───────────────────────────────────────────────────────────────

shap.summary_plot(sv, X_test_sc, feature_names=feature_names, show=False)
plt.title("SHAP Beeswarm — LR (German Credit)")
plt.tight_layout()
plt.savefig("shap_beeswarm_lr_german.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved -> shap_beeswarm_lr_german.png")

# ── 6. Individual explanations ────────────────────────────────────────────────

sorted_idx = np.argsort(y_proba)
high_idx   = sorted_idx[-5]
low_idx    = sorted_idx[4]
mid_idx    = int(np.argmin(np.abs(y_proba - 0.5)))

for label, idx in [("high_risk", high_idx), ("low_risk", low_idx), ("boundary", mid_idx)]:
    sex = "male" if X_test["Sex_male"].iloc[idx] == 1 else "female"
    print(f"\n── {label} (prob={y_proba[idx]:.4f}, sex={sex}) ──────────────────────")
    row_df = pd.DataFrame({
        "feature"   : feature_names,
        "shap_value": sv[idx],
        "feat_value": X_test_sc[idx]
    }).sort_values("shap_value", key=abs, ascending=False)
    print(row_df.to_string(index=False))

    shap.waterfall_plot(
        shap.Explanation(
            values       = sv[idx],
            base_values  = base_val,
            data         = X_test_sc[idx],
            feature_names= feature_names
        ), show=False
    )
    plt.title(f"SHAP Waterfall — {label} LR German Credit\n(sex={sex}, pred={y_proba[idx]:.3f})")
    plt.tight_layout()
    plt.savefig(f"shap_waterfall_{label}_lr_german.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved -> shap_waterfall_{label}_lr_german.png")

print(
    "\n── Thesis fairness summary ─────────────────────────────────────────────\n"
    "Sex_male ranks 6th in global SHAP importance (0.133).\n"
    "Every female applicant receives +0.182 SHAP on sex alone.\n"
    "Every male applicant receives −0.112 SHAP on sex alone.\n"
    "Counterfactual gap: +6.18pp default probability purely from being female.\n"
    "SHAP makes this discrimination visible and quantifiable — a core\n"
    "argument for XAI in regulated credit scoring environments.\n"
    "\nNext: does RF/XGBoost encode the same bias through interactions\n"
    "even when sex contributes less to the global SHAP ranking?\n"
    "Compare sex SHAP ranks across all three models in Chapter 4."
)

print("\nAll LR SHAP outputs saved. Proceed to 04_lime_lr_german.py")

Model retrained.
Base value: 0.0422

── Global mean |SHAP| ──────────────────────────────────────────────────
                    feature  mean_abs_shap
           Checking account       0.502998
                   Duration       0.494040
                Purpose_car       0.147845
Purpose_furniture/equipment       0.144121
                        Age       0.133229
                   Sex_male       0.133106
          Purpose_education       0.120157
                    Housing       0.095512
           Purpose_radio/TV       0.081217
    Purpose_vacation/others       0.068602
            Saving accounts       0.052276
              Credit amount       0.050110
            Purpose_repairs       0.049722
                        Job       0.008413
Purpose_domestic appliances       0.007826
Saved -> shap_global_lr_german.png

── Sex_male SHAP (fairness analysis) ───────────────────────────────────
Rank: 6 of 15
Male   SHAP = -0.1116  (n=139)
Female SHAP = 0.1822  (n=61)
Male   default rate

In [ ]:
pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 14.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=f6a8ed2536eda52e49343fe2de5881c5bddc3fe92fe3190b3e93bc6f2a21b2a2
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [ ]:
"""
04_lime_lr_german.py
====================
LIME explanation of Logistic Regression on German Credit dataset.
Thesis: Evaluating the Trade-off Between Predictive Accuracy and
        Interpretability in Credit Scoring (XAI Comparative Study)

KEY FINDING — LIME vs SHAP on Sex Feature
------------------------------------------
SHAP ranks Sex_male #6 globally (mean |SHAP|=0.133).
LIME ranks Sex_male #13 globally (mean |weight|=0.034) — a 7-rank divergence.

LIME's local neighbourhood sampling perturbs the binary Sex_male feature,
creating mixed-sex artificial instances. This dilutes the constant sex
penalty into noise. SHAP's exact decomposition captures it precisely.

This divergence is a core thesis argument: SHAP is the superior fairness
audit tool for regulated credit scoring. LIME would have nearly missed
the most legally significant variable in the model.

Surrogate R² (German Credit LR):
  High-risk : 0.673  (best of any model/dataset so far)
  Low-risk  : 0.520
  Boundary  : 0.531
  → Strong fit, consistent with LR's linear structure

Dependencies
------------
pip install scikit-learn imbalanced-learn lime pandas numpy matplotlib
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from lime.lime_tabular import LimeTabularExplainer
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE  = 42
TEST_SIZE     = 0.2
N_LIME_LOCAL  = 5000
N_LIME_GLOBAL = 150
N_LIME_AGG    = 2000

# ── 1. Rebuild model ──────────────────────────────────────────────────────────

df = pd.read_csv("german_credit_cleaned.csv", index_col=0)
X  = df.drop("Risk", axis=1)
y  = df["Risk"]
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

sm           = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = sm.fit_resample(X_train_sc, y_train)

lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_res, y_res)
y_proba = lr.predict_proba(X_test_sc)[:, 1]
print("Model retrained.")

# ── 2. LIME explainer ─────────────────────────────────────────────────────────

explainer = LimeTabularExplainer(
    X_res,
    feature_names = feature_names,
    class_names   = ["Good", "Bad"],
    mode          = "classification",
    random_state  = RANDOM_STATE
)

# ── 3. Select representative samples ─────────────────────────────────────────

sorted_idx = np.argsort(y_proba)
high_idx   = sorted_idx[-5]
low_idx    = sorted_idx[4]
mid_idx    = int(np.argmin(np.abs(y_proba - 0.5)))

for label, idx in [("High-risk", high_idx), ("Low-risk", low_idx), ("Boundary", mid_idx)]:
    sex = "male" if X_test["Sex_male"].iloc[idx] == 1 else "female"
    print(f"{label}: prob={y_proba[idx]:.4f}  sex={sex}")

# ── 4. Individual LIME explanations ──────────────────────────────────────────

def explain_and_plot(idx, label, filename):
    exp = explainer.explain_instance(
        X_test_sc[idx], lr.predict_proba,
        num_features=10, num_samples=N_LIME_LOCAL
    )
    available_labels = list(exp.local_exp.keys())
    use_label = 1 if 1 in available_labels else available_labels[-1]
    lime_list = exp.as_list(label=use_label)
    r2        = exp.score
    sex       = "male" if X_test["Sex_male"].iloc[idx] == 1 else "female"

    print(f"\n── {label} (prob={y_proba[idx]:.4f}, R²={r2:.4f}, sex={sex}) ──────────")
    for feat_str, weight in lime_list:
        flag = "  <-- SEX FEATURE" if "Sex_male" in feat_str else ""
        print(f"  {feat_str:<55} {weight:+.4f}{flag}")

    feats   = [f for f, _ in lime_list]
    weights = [w for _, w in lime_list]
    colors  = ["#d85a30" if w > 0 else "#2a7a3b" for w in weights]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(feats[::-1], weights[::-1], color=colors[::-1])
    ax.axvline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.set_xlabel("LIME weight")
    ax.set_title(f"LIME — {label} LR German Credit\npred={y_proba[idx]:.3f}  R²={r2:.3f}  sex={sex}")
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.close()

    return r2

high_r2 = explain_and_plot(high_idx, "HIGH-RISK", "lime_high_risk_lr_german.png")
low_r2  = explain_and_plot(low_idx,  "LOW-RISK",  "lime_low_risk_lr_german.png")
mid_r2  = explain_and_plot(mid_idx,  "BOUNDARY",  "lime_boundary_lr_german.png")

# ── 5. Global aggregation + sex tracking ─────────────────────────────────────

print(f"\n── Aggregating LIME over {N_LIME_GLOBAL} samples ──────────────────────────────")
np.random.seed(RANDOM_STATE)
sample_idx = np.random.choice(len(X_test_sc), N_LIME_GLOBAL, replace=False)

agg = {f: [] for f in feature_names}
sex_weights_male   = []
sex_weights_female = []

for count, i in enumerate(sample_idx):
    if count % 30 == 0:
        print(f"  {count}/{N_LIME_GLOBAL}...")
    exp = explainer.explain_instance(
        X_test_sc[i], lr.predict_proba,
        num_features=10, num_samples=N_LIME_AGG
    )
    available_labels = list(exp.local_exp.keys())
    use_label = 1 if 1 in available_labels else available_labels[-1]
    for feat_str, weight in exp.as_list(label=use_label):
        for fname in feature_names:
            if fname in feat_str:
                agg[fname].append(abs(weight))
                if fname == "Sex_male":
                    if X_test["Sex_male"].iloc[i] == 1:
                        sex_weights_male.append(weight)
                    else:
                        sex_weights_female.append(weight)
                break

agg_df = pd.DataFrame([
    {"feature": k, "mean_abs_weight": np.mean(v) if v else 0.0}
    for k, v in agg.items()
]).sort_values("mean_abs_weight", ascending=False).reset_index(drop=True)

print("\n── Global LIME importance ──────────────────────────────────────────────")
print(agg_df.to_string(index=False))

# ── 6. LIME vs SHAP rank comparison ──────────────────────────────────────────
#
# This is a core thesis result. Sex_male diverges 7 ranks between LIME and SHAP.
# SHAP: #6 (mean |SHAP|=0.133) — captures the constant sex penalty exactly.
# LIME: #13 (mean |weight|=0.034) — sampling dilutes the binary feature effect.
# A regulator relying only on LIME would underweight the sex discrimination signal.

shap_importance = {
    "Checking account"              : 0.503,
    "Duration"                      : 0.494,
    "Purpose_car"                   : 0.148,
    "Purpose_furniture/equipment"   : 0.144,
    "Age"                           : 0.133,
    "Sex_male"                      : 0.133,
    "Purpose_education"             : 0.120,
    "Housing"                       : 0.096,
    "Purpose_radio/TV"              : 0.081,
    "Purpose_vacation/others"       : 0.069,
    "Saving accounts"               : 0.052,
    "Credit amount"                 : 0.050,
    "Purpose_repairs"               : 0.050,
    "Job"                           : 0.008,
    "Purpose_domestic appliances"   : 0.008,
}

comp_df = agg_df.copy()
comp_df["shap_importance"] = comp_df["feature"].map(shap_importance)
comp_df["lime_rank"]  = comp_df["mean_abs_weight"].rank(ascending=False).astype(int)
comp_df["shap_rank"]  = comp_df["shap_importance"].rank(ascending=False).astype(int)
comp_df["rank_delta"] = (comp_df["lime_rank"] - comp_df["shap_rank"]).abs()

print("\n── LIME vs SHAP rank comparison ────────────────────────────────────────")
print(comp_df[["feature","lime_rank","shap_rank","rank_delta"]].to_string(index=False))
sex_row = comp_df[comp_df["feature"]=="Sex_male"]
print(f"\nSex_male: LIME rank={int(sex_row['lime_rank'].values[0])}  "
      f"SHAP rank={int(sex_row['shap_rank'].values[0])}  "
      f"delta={int(sex_row['rank_delta'].values[0])}")

# ── 7. R² summary plot ───────────────────────────────────────────────────────

r2_data = {
    "High-risk" : {"LR (GmSC)": 0.514, "RF (GmSC)": 0.198, "XGB (GmSC)": 0.067,
                   "LR (German)": high_r2},
    "Low-risk"  : {"LR (GmSC)": 0.419, "RF (GmSC)": 0.549, "XGB (GmSC)": 0.057,
                   "LR (German)": low_r2},
    "Boundary"  : {"LR (GmSC)": 0.409, "RF (GmSC)": 0.073, "XGB (GmSC)": 0.056,
                   "LR (German)": mid_r2},
}
print(f"\n── R² summary ───────────────────────────────────────────────────────────")
print(f"High-risk  R² = {high_r2:.4f}")
print(f"Low-risk   R² = {low_r2:.4f}")
print(f"Boundary   R² = {mid_r2:.4f}")
print("German Credit LR achieves highest LIME R² observed — consistent with")
print("smaller, linear dataset being more approachable for local surrogates.")

print(
    "\nThesis note:\n"
    "  LIME misses the sex bias (rank #13) that SHAP precisely captures (rank #6).\n"
    "  For fairness audits in regulated credit scoring, SHAP is the superior tool.\n"
    "  LIME's local sampling approach is structurally unsuited to detecting\n"
    "  constant binary feature effects — a fundamental limitation for bias detection."
)

print("\nAll LR LIME outputs saved. Proceed to 05_pdp_lr_german.py")

Model retrained.
High-risk: prob=0.8875  sex=male
Low-risk: prob=0.1270  sex=male
Boundary: prob=0.4993  sex=male

── HIGH-RISK (prob=0.8875, R²=0.6726, sex=male) ──────────
  Duration > 0.53                                         +0.2538
  Purpose_education > -0.24                               +0.2183
  Purpose_vacation/others <= -0.09                        +0.1981
  Purpose_repairs <= -0.15                                -0.1869
  0.00 < Checking account <= 1.04                         +0.0838
  Purpose_domestic appliances <= -0.11                    -0.0821
  Purpose_furniture/equipment <= -0.48                    -0.0767
  Housing <= -0.90                                        +0.0537
  Purpose_car <= -0.73                                    -0.0449
  Saving accounts > -0.22                                 -0.0367

── LOW-RISK (prob=0.1270, R²=0.5196, sex=male) ──────────
  Purpose_vacation/others <= -0.09                        +0.2382
  Purpose_education <= -0.24             

# RANDOM FOREST


In [ ]:
"""
06_random_forest_german.py
===========================
Random Forest model for German Credit dataset.
Thesis: Evaluating the Trade-off Between Predictive Accuracy and
        Interpretability in Credit Scoring (XAI Comparative Study)

Results
-------
ROC-AUC : 0.7667  (LR: 0.6635) — +0.103 gain
PR-AUC  : 0.6209  (LR: 0.4837) — +0.137 gain
Bad recall    : 60.0%
Bad precision : 50.0%  (LR: 43.8%)
CV ROC-AUC    : 0.7606 ± 0.0475

Confusion matrix: TN=104  FP=36  FN=24  TP=36

Key thesis note
---------------
Unlike Give Me Some Credit (AUC gap: 0.004), German Credit shows a genuine
+10.3pp AUC improvement for RF over LR. This means there IS a real
accuracy-interpretability tradeoff on this dataset — the thesis question
becomes: is +10pp AUC worth the loss of full interpretability?

Sex_male Gini importance: 0.023 (rank #8 of 15)
LR SHAP ranked it #6. SHAP analysis will determine if RF encodes the
same sex bias through feature interactions even when Sex_male's direct
importance is lower.

Dependencies
------------
pip install scikit-learn imbalanced-learn pandas numpy matplotlib
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score,
    average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TEST_SIZE    = 0.2

# ── 1. Load & split ───────────────────────────────────────────────────────────

df = pd.read_csv("german_credit_cleaned.csv", index_col=0)
X  = df.drop("Risk", axis=1)
y  = df["Risk"]
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

# ── 2. Scale + SMOTE ─────────────────────────────────────────────────────────

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

sm           = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = sm.fit_resample(X_train_sc, y_train)
print(f"After SMOTE — good: {(y_res==0).sum()}  bad: {(y_res==1).sum()}")

# ── 3. Train ──────────────────────────────────────────────────────────────────
#
# min_samples_leaf reduced to 10 (vs 50 in GmSC) — dataset is much smaller
# (1k vs 104k rows) so stricter leaf constraints would prevent meaningful splits.

rf = RandomForestClassifier(
    n_estimators    = 300,
    max_depth       = 10,
    min_samples_leaf= 10,
    class_weight    = "balanced",
    random_state    = RANDOM_STATE,
    n_jobs          = -1
)
rf.fit(X_res, y_res)
print("RF trained.")

# ── 4. Evaluate ───────────────────────────────────────────────────────────────

y_pred  = rf.predict(X_test_sc)
y_proba = rf.predict_proba(X_test_sc)[:, 1]

roc_auc = roc_auc_score(y_test, y_proba)
pr_auc  = average_precision_score(y_test, y_proba)

print(f"\nROC-AUC : {roc_auc:.4f}  (LR: 0.6635, delta: {roc_auc-0.6635:+.4f})")
print(f"PR-AUC  : {pr_auc:.4f}  (LR: 0.4837, delta: {pr_auc-0.4837:+.4f})")
print(classification_report(y_test, y_pred, target_names=["Good","Bad"]))

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

# ── 5. Cross-validation ───────────────────────────────────────────────────────

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(rf, X_train_sc, y_train, cv=cv, scoring="roc_auc")
print(f"\n5-fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# ── 6. Feature importance ─────────────────────────────────────────────────────

fi_df = pd.DataFrame({
    "feature"   : feature_names,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("\nGini importance:")
print(fi_df.to_string(index=False))
sex_rank = fi_df[fi_df["feature"]=="Sex_male"].index[0] + 1
print(f"\nSex_male rank: #{sex_rank} (LR SHAP rank was #6)")

# ── 7. Plots ──────────────────────────────────────────────────────────────────

fpr, tpr, _          = roc_curve(y_test, y_proba)
precision, recall, _ = precision_recall_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color="#2a7a3b", lw=2, label=f"RF (AUC={roc_auc:.3f})")
ax.plot([0,1],[0,1], color="gray", lw=1, linestyle="--")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC — RF (German Credit)"); ax.legend()
plt.tight_layout(); plt.savefig("roc_rf_german.png", dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recall, precision, color="#2a7a3b", lw=2, label=f"RF (AP={pr_auc:.3f})")
ax.axhline(y=y_test.mean(), color="gray", lw=1, linestyle="--")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("PR Curve — RF (German Credit)"); ax.legend()
plt.tight_layout(); plt.savefig("pr_rf_german.png", dpi=150); plt.close()

colors = ["#d85a30" if f == "Sex_male" else "#2a7a3b" for f in fi_df["feature"][::-1]]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1], color=colors)
ax.set_xlabel("Gini importance")
ax.set_title("RF Feature Importance — German Credit\n(red = Sex_male)")
plt.tight_layout(); plt.savefig("fi_rf_german.png", dpi=150); plt.close()

print("\nSaved -> roc_rf_german.png, pr_rf_german.png, fi_rf_german.png")
print("Proceed to 07_shap_rf_german.py")

After SMOTE — good: 560  bad: 560
RF trained.

ROC-AUC : 0.7667  (LR: 0.6635, delta: +0.1032)
PR-AUC  : 0.6209  (LR: 0.4837, delta: +0.1372)
              precision    recall  f1-score   support

        Good       0.81      0.74      0.78       140
         Bad       0.50      0.60      0.55        60

    accuracy                           0.70       200
   macro avg       0.66      0.67      0.66       200
weighted avg       0.72      0.70      0.71       200

TN=104  FP=36  FN=24  TP=36

5-fold CV ROC-AUC: 0.7606 ± 0.0475

Gini importance:
                    feature  importance
           Checking account    0.374179
                   Duration    0.181340
            Saving accounts    0.118588
              Credit amount    0.103287
                        Age    0.076638
                    Housing    0.039437
                        Job    0.032692
                   Sex_male    0.023366
           Purpose_radio/TV    0.019561
                Purpose_car    0.016864
          

In [ ]:
"""
07_shap_rf_german.py
====================
SHAP TreeExplainer analysis for Random Forest on German Credit.
Thesis: Evaluating the Trade-off Between Predictive Accuracy and
        Interpretability in Credit Scoring (XAI Comparative Study)

KEY FAIRNESS FINDING — Sex Bias Reduced but Not Eliminated
-----------------------------------------------------------
                    LR              RF
Sex_male SHAP rank: #6              #8
Mean |SHAP|:        0.133           0.014   (10x reduction)
Male mean SHAP:     −0.112          −0.012
Female mean SHAP:   +0.182          +0.019
Counterfactual gap: +6.18pp         +2.68pp (57% reduction)

RF reduces but does not eliminate sex discrimination.
The signal is redistributed across feature interactions — notably
Age × Sex_male (r=0.269), the strongest SHAP correlation.

This is a critical thesis argument: RF's "fairness improvement" is
superficial — the bias is buried in interactions, making it harder
to detect and audit. SHAP is the only tool that exposes this.

Dependencies
------------
pip install scikit-learn imbalanced-learn shap pandas numpy matplotlib
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import shap
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TEST_SIZE    = 0.2

# ── 1. Rebuild model ──────────────────────────────────────────────────────────

df = pd.read_csv("german_credit_cleaned.csv", index_col=0)
X  = df.drop("Risk", axis=1)
y  = df["Risk"]
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

sm           = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = sm.fit_resample(X_train_sc, y_train)

rf = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=10,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_res, y_res)
feature_names = list(X.columns)
print("Model retrained.")

# ── 2. TreeExplainer ──────────────────────────────────────────────────────────

explainer   = shap.TreeExplainer(rf)
sv_all      = explainer.shap_values(X_test_sc)  # (200, 15, 2)
sv          = sv_all[:, :, 1]                   # class 1 = bad/default
base_val    = explainer.expected_value[1]
y_proba     = rf.predict_proba(X_test_sc)[:, 1]
print(f"Base value: {base_val:.4f}")

# ── 3. Global importance ──────────────────────────────────────────────────────

mean_abs = np.abs(sv).mean(axis=0)
imp_df   = pd.DataFrame({
    "feature"      : feature_names,
    "mean_abs_shap": mean_abs
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

print("\n── Global mean |SHAP| ──────────────────────────────────────────────────")
print(imp_df.to_string(index=False))

# ── 4. Sex_male fairness analysis ─────────────────────────────────────────────

sex_idx   = feature_names.index("Sex_male")
male_mask = X_test["Sex_male"].values == 1

print("\n── Sex_male SHAP fairness analysis ─────────────────────────────────────")
sex_rank = imp_df[imp_df["feature"]=="Sex_male"].index[0] + 1
print(f"Rank: #{sex_rank}  Mean |SHAP|: {mean_abs[sex_idx]:.4f}  (LR: 0.133)")
print(f"Male SHAP  (n={male_mask.sum()}):   {sv[male_mask, sex_idx].mean():.4f}  (LR: −0.112)")
print(f"Female SHAP (n={(~male_mask).sum()}): {sv[~male_mask, sex_idx].mean():.4f}   (LR: +0.182)")

# Counterfactual analysis
X_cf_male   = X_test_sc.copy(); X_cf_male[:, sex_idx]   = X_test_sc[:, sex_idx].max()
X_cf_female = X_test_sc.copy(); X_cf_female[:, sex_idx] = X_test_sc[:, sex_idx].min()
gap_rf = rf.predict_proba(X_cf_female)[:,1].mean() - rf.predict_proba(X_cf_male)[:,1].mean()
gap_lr = 0.0618
print(f"\nCounterfactual gap:")
print(f"  LR: +{gap_lr*100:.2f}pp  (unconditional, constant for all borrowers)")
print(f"  RF: +{gap_rf*100:.2f}pp  (57% reduction — but still present)")
print(f"\nInterpretation: RF buried the sex bias in feature interactions.")
print(f"Age × Sex_male SHAP correlation = 0.269 — older female borrowers")
print(f"are disproportionately affected. The bias is context-dependent,")
print(f"making it harder to detect, audit, and challenge.")

# ── 5. Beeswarm ───────────────────────────────────────────────────────────────

shap.summary_plot(sv, X_test_sc, feature_names=feature_names, show=False)
plt.title("SHAP Beeswarm — RF (German Credit)")
plt.tight_layout()
plt.savefig("shap_beeswarm_rf_german.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nSaved -> shap_beeswarm_rf_german.png")

# ── 6. Individual waterfalls ──────────────────────────────────────────────────

sorted_idx = np.argsort(y_proba)
high_idx   = sorted_idx[-5]
low_idx    = sorted_idx[4]
mid_idx    = int(np.argmin(np.abs(y_proba - 0.5)))

for label, idx in [("high_risk", high_idx), ("low_risk", low_idx), ("boundary", mid_idx)]:
    sex = "male" if X_test["Sex_male"].iloc[idx] == 1 else "female"
    print(f"\n── {label} (prob={y_proba[idx]:.4f}, sex={sex}) ──────────────────────")
    row_df = pd.DataFrame({
        "feature"   : feature_names,
        "shap_value": sv[idx],
    }).sort_values("shap_value", key=abs, ascending=False)
    print(row_df.to_string(index=False))

    shap.waterfall_plot(
        shap.Explanation(
            values       = sv[idx],
            base_values  = base_val,
            data         = X_test_sc[idx],
            feature_names= feature_names
        ), show=False
    )
    plt.title(f"SHAP Waterfall — {label} RF German Credit\n(sex={sex})")
    plt.tight_layout()
    plt.savefig(f"shap_waterfall_{label}_rf_german.png", dpi=150, bbox_inches="tight")
    plt.close()

# ── 7. LR vs RF comparison ────────────────────────────────────────────────────

lr_shap_imp = {
    "Checking account": 0.503, "Duration": 0.494, "Purpose_car": 0.148,
    "Purpose_furniture/equipment": 0.144, "Age": 0.133, "Sex_male": 0.133,
    "Purpose_education": 0.120, "Housing": 0.096, "Purpose_radio/TV": 0.081,
    "Purpose_vacation/others": 0.069, "Saving accounts": 0.052,
    "Credit amount": 0.050, "Purpose_repairs": 0.050,
    "Job": 0.008, "Purpose_domestic appliances": 0.008,
}
comp_df = imp_df.copy()
comp_df["lr_shap"]  = comp_df["feature"].map(lr_shap_imp)
comp_df["lr_rank"]  = comp_df["lr_shap"].rank(ascending=False).astype(int)
comp_df["rf_rank"]  = comp_df["mean_abs_shap"].rank(ascending=False).astype(int)
comp_df["rank_delta"] = (comp_df["rf_rank"] - comp_df["lr_rank"]).abs()

print("\n── LR vs RF SHAP rank comparison ──────────────────────────────────────")
print(comp_df[["feature","lr_rank","rf_rank","rank_delta","lr_shap","mean_abs_shap"]].to_string(index=False))

# ── 8. Interaction proxy ──────────────────────────────────────────────────────

sv_df  = pd.DataFrame(sv, columns=feature_names)
corr   = sv_df.corr()
pairs  = [(feature_names[i], feature_names[j], corr.iloc[i,j])
          for i in range(len(feature_names))
          for j in range(i+1, len(feature_names))
          if not np.isnan(corr.iloc[i,j])]
pairs.sort(key=lambda x: abs(x[2]), reverse=True)
print("\n── Top SHAP correlations (interactions) ────────────────────────────────")
for a, b, c in pairs[:5]:
    print(f"  {a[:35]} x {b[:35]}  r={c:.3f}")

print(
    "\nThesis summary:\n"
    f"  RF sex counterfactual gap: +{gap_rf*100:.2f}pp (LR: +6.18pp)\n"
    "  57% reduction — but bias is NOT eliminated.\n"
    "  Age × Sex_male interaction (r=0.269) suggests RF\n"
    "  treats older female borrowers differently — a subtler\n"
    "  but equally problematic form of discrimination that\n"
    "  SHAP uniquely surfaces among XAI methods."
)
print("\nAll RF SHAP outputs saved. Proceed to 08_lime_rf_german.py")

Model retrained.
Base value: 0.5000

── Global mean |SHAP| ──────────────────────────────────────────────────
                    feature  mean_abs_shap
           Checking account       0.136509
                   Duration       0.079388
            Saving accounts       0.044408
              Credit amount       0.024345
                    Housing       0.023311
                        Age       0.017115
           Purpose_radio/TV       0.015783
                   Sex_male       0.013903
          Purpose_education       0.006294
                        Job       0.006249
                Purpose_car       0.005725
Purpose_furniture/equipment       0.001923
            Purpose_repairs       0.000266
Purpose_domestic appliances       0.000000
    Purpose_vacation/others       0.000000

── Sex_male SHAP fairness analysis ─────────────────────────────────────
Rank: #8  Mean |SHAP|: 0.0139  (LR: 0.133)
Male SHAP  (n=139):   -0.0116  (LR: −0.112)
Female SHAP (n=61): 0.0187   (LR: +0.182)

In [ ]:
"""
08_lime_rf_german.py
====================
LIME explanation of Random Forest on German Credit.
Thesis: Evaluating the Trade-off Between Predictive Accuracy and
        Interpretability in Credit Scoring (XAI Comparative Study)

KEY FINDINGS
------------
1. R² ASYMMETRY (new finding):
   High-risk R²  = 0.215  (poor — RF complex around high-risk borrowers)
   Low-risk R²   = 0.834  (excellent — best across all models/datasets)
   Boundary R²   = 0.841  (excellent — highest boundary R² observed)

   This asymmetry is absent in LR (R²≈0.5 consistently). The practical
   implication: LIME explanations are reliable for APPROVED applicants
   but unreliable for REJECTED ones — backwards from regulatory need.

2. CHECKING ACCOUNT DIVERGENCE:
   LIME ranks checking account #10 (0.018). SHAP ranks it #1 (0.137).
   A 9-rank divergence on the most important feature. LIME's local
   sampling focused on duration instead — a fundamentally different
   explanation of RF's decision-making.

3. SEX CONSISTENTLY LOW:
   LIME sex rank: #14. SHAP sex rank: #8.
   Both methods agree sex is less important in RF than in LR.
   LIME male weight: −0.012, female weight: +0.012 — consistent direction
   but much smaller than LR's −0.112/+0.182.

Dependencies
------------
pip install scikit-learn imbalanced-learn lime pandas numpy matplotlib
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from lime.lime_tabular import LimeTabularExplainer
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE  = 42
TEST_SIZE     = 0.2
N_LIME_LOCAL  = 5000
N_LIME_GLOBAL = 150
N_LIME_AGG    = 2000

# ── 1. Rebuild model ──────────────────────────────────────────────────────────

df = pd.read_csv("german_credit_cleaned.csv", index_col=0)
X  = df.drop("Risk", axis=1)
y  = df["Risk"]
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

sm           = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = sm.fit_resample(X_train_sc, y_train)

rf = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=10,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_res, y_res)
y_proba = rf.predict_proba(X_test_sc)[:, 1]
print("Model retrained.")

# ── 2. LIME explainer ─────────────────────────────────────────────────────────

explainer = LimeTabularExplainer(
    X_res, feature_names=feature_names,
    class_names=["Good","Bad"], mode="classification",
    random_state=RANDOM_STATE
)

# ── 3. Representative samples ─────────────────────────────────────────────────

sorted_idx = np.argsort(y_proba)
high_idx   = sorted_idx[-5]
low_idx    = sorted_idx[4]
mid_idx    = int(np.argmin(np.abs(y_proba - 0.5)))

# ── 4. Individual explanations ────────────────────────────────────────────────

def explain_and_plot(idx, label, filename):
    exp = explainer.explain_instance(
        X_test_sc[idx], rf.predict_proba,
        num_features=10, num_samples=N_LIME_LOCAL
    )
    use_label = 1 if 1 in exp.local_exp else list(exp.local_exp.keys())[-1]
    lime_list = exp.as_list(label=use_label)
    r2        = exp.score
    sex       = "male" if X_test["Sex_male"].iloc[idx] == 1 else "female"

    print(f"\n── {label} (prob={y_proba[idx]:.4f}, R²={r2:.4f}, sex={sex}) ──────────")
    for feat_str, weight in lime_list:
        flag = "  <-- SEX" if "Sex_male" in feat_str else ""
        print(f"  {feat_str:<55} {weight:+.4f}{flag}")

    if r2 < 0.3:
        print(f"  ⚠ R²={r2:.3f} — explanation unreliable")

    feats   = [f for f, _ in lime_list]
    weights = [w for _, w in lime_list]
    colors  = ["#d85a30" if w > 0 else "#2a7a3b" for w in weights]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(feats[::-1], weights[::-1], color=colors[::-1])
    ax.axvline(0, color="gray", lw=0.8, linestyle="--")
    ax.set_xlabel("LIME weight")
    ax.set_title(f"LIME — {label} RF German Credit\npred={y_proba[idx]:.3f}  R²={r2:.3f}  sex={sex}"
                 + ("  ⚠ LOW R²" if r2 < 0.3 else ""))
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.close()
    return r2

high_r2 = explain_and_plot(high_idx, "HIGH-RISK", "lime_high_risk_rf_german.png")
low_r2  = explain_and_plot(low_idx,  "LOW-RISK",  "lime_low_risk_rf_german.png")
mid_r2  = explain_and_plot(mid_idx,  "BOUNDARY",  "lime_boundary_rf_german.png")

# ── 5. Global aggregation ─────────────────────────────────────────────────────

print(f"\n── Aggregating LIME over {N_LIME_GLOBAL} samples ──────────────────────────────")
np.random.seed(RANDOM_STATE)
sample_idx = np.random.choice(len(X_test_sc), N_LIME_GLOBAL, replace=False)

agg = {f: [] for f in feature_names}
sex_male_w, sex_female_w = [], []

for count, i in enumerate(sample_idx):
    if count % 30 == 0:
        print(f"  {count}/{N_LIME_GLOBAL}...")
    exp = explainer.explain_instance(
        X_test_sc[i], rf.predict_proba,
        num_features=10, num_samples=N_LIME_AGG
    )
    use_label = 1 if 1 in exp.local_exp else list(exp.local_exp.keys())[-1]
    for feat_str, weight in exp.as_list(label=use_label):
        for fname in feature_names:
            if fname in feat_str:
                agg[fname].append(abs(weight))
                if fname == "Sex_male":
                    if X_test["Sex_male"].iloc[i] == 1:
                        sex_male_w.append(weight)
                    else:
                        sex_female_w.append(weight)
                break

agg_df = pd.DataFrame([
    {"feature": k, "mean_abs_weight": np.mean(v) if v else 0.0}
    for k, v in agg.items()
]).sort_values("mean_abs_weight", ascending=False).reset_index(drop=True)

print("\n── Global LIME importance ──────────────────────────────────────────────")
print(agg_df.to_string(index=False))
print(f"\nSex_male: male weight={np.mean(sex_male_w):.4f}  female weight={np.mean(sex_female_w):.4f}")

# ── 6. LIME vs SHAP rank comparison ──────────────────────────────────────────

shap_importance = {
    "Checking account": 0.137, "Duration": 0.079, "Saving accounts": 0.044,
    "Credit amount": 0.024, "Housing": 0.023, "Age": 0.017,
    "Purpose_radio/TV": 0.016, "Sex_male": 0.014, "Purpose_education": 0.006,
    "Job": 0.006, "Purpose_car": 0.006, "Purpose_furniture/equipment": 0.002,
    "Purpose_repairs": 0.000, "Purpose_domestic appliances": 0.000,
    "Purpose_vacation/others": 0.000,
}
comp_df = agg_df.copy()
comp_df["shap_imp"]   = comp_df["feature"].map(shap_importance)
comp_df["lime_rank"]  = comp_df["mean_abs_weight"].rank(ascending=False).astype(int)
comp_df["shap_rank"]  = comp_df["shap_imp"].rank(ascending=False).astype(int)
comp_df["rank_delta"] = (comp_df["lime_rank"] - comp_df["shap_rank"]).abs()

print("\n── LIME vs SHAP rank comparison ────────────────────────────────────────")
print(comp_df[["feature","lime_rank","shap_rank","rank_delta"]].to_string(index=False))

# ── 7. R² comparison plot ─────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(7, 5))
labels  = ["High-risk","Low-risk","Boundary"]
lr_r2   = [0.673, 0.520, 0.531]
rf_r2   = [high_r2, low_r2, mid_r2]
x = np.arange(3); w = 0.35
ax.bar(x-w/2, lr_r2, w, label="LR", color="#3266ad")
ax.bar(x+w/2, rf_r2, w, label="RF", color="#2a7a3b")
ax.axhline(0.5, color="#d85a30", linestyle="--", lw=1.5, label="R²=0.5 threshold")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0, 1); ax.set_ylabel("LIME Surrogate R²")
ax.set_title("LIME R² — LR vs RF (German Credit)\nRF asymmetric: excellent for safe, poor for risky borrowers")
ax.legend()
plt.tight_layout()
plt.savefig("lime_r2_lr_vs_rf_german.png", dpi=150)
plt.close()
print("\nSaved -> lime_r2_lr_vs_rf_german.png")

print(
    "\nThesis summary:\n"
    "  R² asymmetry: RF excellent for low/boundary (0.83-0.84) but poor for high-risk (0.22)\n"
    "  This means LIME explanations are unreliable for REJECTED applicants.\n"
    "  Checking account: LIME #10 vs SHAP #1 — 9-rank divergence.\n"
    "  Sex: both LIME and SHAP agree it is less important in RF than LR.\n"
    "  But neither tool captures the full interaction effect (Age × Sex).\n"
    "  Conclusion: SHAP + PDP remain essential complements to LIME."
)
print("\nAll RF LIME outputs saved. Proceed to 09_pdp_rf_german.py")

Model retrained.

── HIGH-RISK (prob=0.7837, R²=0.2149, sex=male) ──────────
  Duration > 0.53                                         +0.1153
  Purpose_radio/TV <= -0.61                               +0.0716
  Saving accounts <= -0.22                                +0.0397
  Age > 0.48                                              -0.0314
  Job > 0.15                                              -0.0260
  Purpose_vacation/others > -0.09                         +0.0238
  Purpose_education <= -0.24                              -0.0188
  -1.49 < Sex_male <= 0.67                                -0.0108  <-- SEX
  Purpose_repairs <= -0.15                                +0.0107
  Purpose_domestic appliances <= -0.11                    -0.0095
  ⚠ R²=0.215 — explanation unreliable

── LOW-RISK (prob=0.0986, R²=0.8340, sex=male) ──────────
  Duration <= -0.74                                       -0.2705
  Purpose_radio/TV <= -0.61                               +0.0630
  Saving accounts <= -0.2

In [ ]:
"""
08_lime_rf_german.py
====================
LIME explanation of Random Forest on German Credit.
Thesis: Evaluating the Trade-off Between Predictive Accuracy and
        Interpretability in Credit Scoring (XAI Comparative Study)

KEY FINDINGS
------------
1. R² ASYMMETRY (new finding):
   High-risk R²  = 0.215  (poor — RF complex around high-risk borrowers)
   Low-risk R²   = 0.834  (excellent — best across all models/datasets)
   Boundary R²   = 0.841  (excellent — highest boundary R² observed)

   This asymmetry is absent in LR (R²≈0.5 consistently). The practical
   implication: LIME explanations are reliable for APPROVED applicants
   but unreliable for REJECTED ones — backwards from regulatory need.

2. CHECKING ACCOUNT DIVERGENCE:
   LIME ranks checking account #10 (0.018). SHAP ranks it #1 (0.137).
   A 9-rank divergence on the most important feature. LIME's local
   sampling focused on duration instead — a fundamentally different
   explanation of RF's decision-making.

3. SEX CONSISTENTLY LOW:
   LIME sex rank: #14. SHAP sex rank: #8.
   Both methods agree sex is less important in RF than in LR.
   LIME male weight: −0.012, female weight: +0.012 — consistent direction
   but much smaller than LR's −0.112/+0.182.

Dependencies
------------
pip install scikit-learn imbalanced-learn lime pandas numpy matplotlib
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from lime.lime_tabular import LimeTabularExplainer
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE  = 42
TEST_SIZE     = 0.2
N_LIME_LOCAL  = 5000
N_LIME_GLOBAL = 150
N_LIME_AGG    = 2000

# ── 1. Rebuild model ──────────────────────────────────────────────────────────

df = pd.read_csv("german_credit_cleaned.csv", index_col=0)
X  = df.drop("Risk", axis=1)
y  = df["Risk"]
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

sm           = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = sm.fit_resample(X_train_sc, y_train)

rf = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=10,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_res, y_res)
y_proba = rf.predict_proba(X_test_sc)[:, 1]
print("Model retrained.")

# ── 2. LIME explainer ─────────────────────────────────────────────────────────

explainer = LimeTabularExplainer(
    X_res, feature_names=feature_names,
    class_names=["Good","Bad"], mode="classification",
    random_state=RANDOM_STATE
)

# ── 3. Representative samples ─────────────────────────────────────────────────

sorted_idx = np.argsort(y_proba)
high_idx   = sorted_idx[-5]
low_idx    = sorted_idx[4]
mid_idx    = int(np.argmin(np.abs(y_proba - 0.5)))

# ── 4. Individual explanations ────────────────────────────────────────────────

def explain_and_plot(idx, label, filename):
    exp = explainer.explain_instance(
        X_test_sc[idx], rf.predict_proba,
        num_features=10, num_samples=N_LIME_LOCAL
    )
    use_label = 1 if 1 in exp.local_exp else list(exp.local_exp.keys())[-1]
    lime_list = exp.as_list(label=use_label)
    r2        = exp.score
    sex       = "male" if X_test["Sex_male"].iloc[idx] == 1 else "female"

    print(f"\n── {label} (prob={y_proba[idx]:.4f}, R²={r2:.4f}, sex={sex}) ──────────")
    for feat_str, weight in lime_list:
        flag = "  <-- SEX" if "Sex_male" in feat_str else ""
        print(f"  {feat_str:<55} {weight:+.4f}{flag}")

    if r2 < 0.3:
        print(f"  ⚠ R²={r2:.3f} — explanation unreliable")

    feats   = [f for f, _ in lime_list]
    weights = [w for _, w in lime_list]
    colors  = ["#d85a30" if w > 0 else "#2a7a3b" for w in weights]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(feats[::-1], weights[::-1], color=colors[::-1])
    ax.axvline(0, color="gray", lw=0.8, linestyle="--")
    ax.set_xlabel("LIME weight")
    ax.set_title(f"LIME — {label} RF German Credit\npred={y_proba[idx]:.3f}  R²={r2:.3f}  sex={sex}"
                 + ("  ⚠ LOW R²" if r2 < 0.3 else ""))
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.close()
    return r2

high_r2 = explain_and_plot(high_idx, "HIGH-RISK", "lime_high_risk_rf_german.png")
low_r2  = explain_and_plot(low_idx,  "LOW-RISK",  "lime_low_risk_rf_german.png")
mid_r2  = explain_and_plot(mid_idx,  "BOUNDARY",  "lime_boundary_rf_german.png")

# ── 5. Global aggregation ─────────────────────────────────────────────────────

print(f"\n── Aggregating LIME over {N_LIME_GLOBAL} samples ──────────────────────────────")
np.random.seed(RANDOM_STATE)
sample_idx = np.random.choice(len(X_test_sc), N_LIME_GLOBAL, replace=False)

agg = {f: [] for f in feature_names}
sex_male_w, sex_female_w = [], []

for count, i in enumerate(sample_idx):
    if count % 30 == 0:
        print(f"  {count}/{N_LIME_GLOBAL}...")
    exp = explainer.explain_instance(
        X_test_sc[i], rf.predict_proba,
        num_features=10, num_samples=N_LIME_AGG
    )
    use_label = 1 if 1 in exp.local_exp else list(exp.local_exp.keys())[-1]
    for feat_str, weight in exp.as_list(label=use_label):
        for fname in feature_names:
            if fname in feat_str:
                agg[fname].append(abs(weight))
                if fname == "Sex_male":
                    if X_test["Sex_male"].iloc[i] == 1:
                        sex_male_w.append(weight)
                    else:
                        sex_female_w.append(weight)
                break

agg_df = pd.DataFrame([
    {"feature": k, "mean_abs_weight": np.mean(v) if v else 0.0}
    for k, v in agg.items()
]).sort_values("mean_abs_weight", ascending=False).reset_index(drop=True)

print("\n── Global LIME importance ──────────────────────────────────────────────")
print(agg_df.to_string(index=False))
print(f"\nSex_male: male weight={np.mean(sex_male_w):.4f}  female weight={np.mean(sex_female_w):.4f}")

# ── 6. LIME vs SHAP rank comparison ──────────────────────────────────────────

shap_importance = {
    "Checking account": 0.137, "Duration": 0.079, "Saving accounts": 0.044,
    "Credit amount": 0.024, "Housing": 0.023, "Age": 0.017,
    "Purpose_radio/TV": 0.016, "Sex_male": 0.014, "Purpose_education": 0.006,
    "Job": 0.006, "Purpose_car": 0.006, "Purpose_furniture/equipment": 0.002,
    "Purpose_repairs": 0.000, "Purpose_domestic appliances": 0.000,
    "Purpose_vacation/others": 0.000,
}
comp_df = agg_df.copy()
comp_df["shap_imp"]   = comp_df["feature"].map(shap_importance)
comp_df["lime_rank"]  = comp_df["mean_abs_weight"].rank(ascending=False).astype(int)
comp_df["shap_rank"]  = comp_df["shap_imp"].rank(ascending=False).astype(int)
comp_df["rank_delta"] = (comp_df["lime_rank"] - comp_df["shap_rank"]).abs()

print("\n── LIME vs SHAP rank comparison ────────────────────────────────────────")
print(comp_df[["feature","lime_rank","shap_rank","rank_delta"]].to_string(index=False))

# ── 7. R² comparison plot ─────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(7, 5))
labels  = ["High-risk","Low-risk","Boundary"]
lr_r2   = [0.673, 0.520, 0.531]
rf_r2   = [high_r2, low_r2, mid_r2]
x = np.arange(3); w = 0.35
ax.bar(x-w/2, lr_r2, w, label="LR", color="#3266ad")
ax.bar(x+w/2, rf_r2, w, label="RF", color="#2a7a3b")
ax.axhline(0.5, color="#d85a30", linestyle="--", lw=1.5, label="R²=0.5 threshold")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0, 1); ax.set_ylabel("LIME Surrogate R²")
ax.set_title("LIME R² — LR vs RF (German Credit)\nRF asymmetric: excellent for safe, poor for risky borrowers")
ax.legend()
plt.tight_layout()
plt.savefig("lime_r2_lr_vs_rf_german.png", dpi=150)
plt.close()
print("\nSaved -> lime_r2_lr_vs_rf_german.png")

print(
    "\nThesis summary:\n"
    "  R² asymmetry: RF excellent for low/boundary (0.83-0.84) but poor for high-risk (0.22)\n"
    "  This means LIME explanations are unreliable for REJECTED applicants.\n"
    "  Checking account: LIME #10 vs SHAP #1 — 9-rank divergence.\n"
    "  Sex: both LIME and SHAP agree it is less important in RF than LR.\n"
    "  But neither tool captures the full interaction effect (Age × Sex).\n"
    "  Conclusion: SHAP + PDP remain essential complements to LIME."
)
print("\nAll RF LIME outputs saved. Proceed to 09_pdp_rf_german.py")

Model retrained.

── HIGH-RISK (prob=0.7837, R²=0.2149, sex=male) ──────────
  Duration > 0.53                                         +0.1153
  Purpose_radio/TV <= -0.61                               +0.0716
  Saving accounts <= -0.22                                +0.0397
  Age > 0.48                                              -0.0314
  Job > 0.15                                              -0.0260
  Purpose_vacation/others > -0.09                         +0.0238
  Purpose_education <= -0.24                              -0.0188
  -1.49 < Sex_male <= 0.67                                -0.0108  <-- SEX
  Purpose_repairs <= -0.15                                +0.0107
  Purpose_domestic appliances <= -0.11                    -0.0095
  ⚠ R²=0.215 — explanation unreliable

── LOW-RISK (prob=0.0986, R²=0.8340, sex=male) ──────────
  Duration <= -0.74                                       -0.2705
  Purpose_radio/TV <= -0.61                               +0.0630
  Saving accounts <= -0.2

# XGBOOST

In [ ]:
"""
10_xgboost_german.py
=====================
XGBoost model for German Credit dataset.
Thesis: Evaluating the Trade-off Between Predictive Accuracy and
        Interpretability in Credit Scoring (XAI Comparative Study)

Results
-------
ROC-AUC : 0.7732  (LR: 0.6635 | RF: 0.7667)
PR-AUC  : 0.6143  (LR: 0.4837 | RF: 0.6209)
Bad recall    : 61.7%
Bad precision : 56.9%  — best of all three models
CV ROC-AUC    : 0.7720 ± 0.0504

Confusion matrix: TN=112  FP=28  FN=23  TP=37
Fewest false positives (28) of all three models.

Key thesis note — German Credit is the thesis's clearest accuracy tradeoff
-----------------------------------------------------------------------
LR  ROC-AUC: 0.664  (fully interpretable)
RF  ROC-AUC: 0.767  (+10.3pp over LR)
XGB ROC-AUC: 0.773  (+10.9pp over LR, +0.6pp over RF)

Unlike Give Me Some Credit (all models within 0.007 AUC), German Credit
shows a genuine +11pp gain for complex models. The question for Chapter 4:
is +11pp worth the loss of full interpretability AND the risk of encoding
sex discrimination in harder-to-audit interaction terms?

Hyperparameter note
-------------------
max_depth reduced to 4 (vs 5 in GmSC) — smaller dataset (1k rows) benefits
from shallower trees to avoid overfitting. CV std=0.050 is acceptable.

Dependencies
------------
pip install xgboost scikit-learn imbalanced-learn pandas numpy matplotlib
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score,
    average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TEST_SIZE    = 0.2

# ── 1. Load & split ───────────────────────────────────────────────────────────

df = pd.read_csv("german_credit_cleaned.csv", index_col=0)
X  = df.drop("Risk", axis=1)
y  = df["Risk"]
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

# ── 2. Scale + SMOTE ─────────────────────────────────────────────────────────

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

sm           = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = sm.fit_resample(X_train_sc, y_train)
scale_pos_weight = (y_res==0).sum()/(y_res==1).sum()

print(f"After SMOTE — good: {(y_res==0).sum()}  bad: {(y_res==1).sum()}")

# ── 3. Train ──────────────────────────────────────────────────────────────────

xgb = XGBClassifier(
    n_estimators     = 300,
    max_depth        = 4,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = scale_pos_weight,
    eval_metric      = "logloss",
    random_state     = RANDOM_STATE,
    n_jobs           = -1
)
xgb.fit(X_res, y_res)
print("XGBoost trained.")

# ── 4. Evaluate ───────────────────────────────────────────────────────────────

y_pred  = xgb.predict(X_test_sc)
y_proba = xgb.predict_proba(X_test_sc)[:, 1]

roc_auc = roc_auc_score(y_test, y_proba)
pr_auc  = average_precision_score(y_test, y_proba)

print(f"\nROC-AUC : {roc_auc:.4f}  (LR: 0.6635 | RF: 0.7667)")
print(f"PR-AUC  : {pr_auc:.4f}  (LR: 0.4837 | RF: 0.6209)")
print(classification_report(y_test, y_pred, target_names=["Good","Bad"]))

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

# ── 5. Cross-validation ───────────────────────────────────────────────────────

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(xgb, X_train_sc, y_train, cv=cv, scoring="roc_auc")
print(f"\n5-fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# ── 6. Feature importance (gain) ──────────────────────────────────────────────

fi_raw = xgb.get_booster().get_score(importance_type="gain")
fi_df  = pd.DataFrame([
    {"feature": feature_names[int(k[1:])], "gain": v}
    for k, v in fi_raw.items()
]).sort_values("gain", ascending=False).reset_index(drop=True)

print("\nGain importance:")
print(fi_df.to_string(index=False))
sex_rank = fi_df[fi_df["feature"]=="Sex_male"].index[0] + 1
print(f"\nSex_male rank: #{sex_rank} (LR SHAP: #6 | RF SHAP: #8 | RF Gini: #8)")

# ── 7. Plots ──────────────────────────────────────────────────────────────────

fpr, tpr, _          = roc_curve(y_test, y_proba)
precision, recall, _ = precision_recall_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color="#b07d2a", lw=2, label=f"XGB (AUC={roc_auc:.3f})")
ax.plot([0,1],[0,1], color="gray", lw=1, linestyle="--")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC — XGB (German Credit)"); ax.legend()
plt.tight_layout(); plt.savefig("roc_xgb_german.png", dpi=150); plt.close()

colors = ["#d85a30" if f == "Sex_male" else "#b07d2a" for f in fi_df["feature"][::-1]]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi_df["feature"][::-1], fi_df["gain"][::-1], color=colors)
ax.set_xlabel("Gain")
ax.set_title("XGB Feature Importance (Gain) — German Credit\n(red = Sex_male)")
plt.tight_layout(); plt.savefig("fi_xgb_german.png", dpi=150); plt.close()

print("\nSaved -> roc_xgb_german.png, fi_xgb_german.png")
print("Proceed to 11_shap_xgb_german.py")

In [ ]:
"""
11_shap_xgb_german.py
======================
SHAP TreeExplainer analysis for XGBoost on German Credit.
Thesis: Evaluating the Trade-off Between Predictive Accuracy and
        Interpretability in Credit Scoring (XAI Comparative Study)

KEY FAIRNESS FINDING — Boosting Re-amplifies Sex Bias
-------------------------------------------------------
                    LR          RF          XGB
Sex rank:           #6          #8          #7
Mean |SHAP|:        0.133       0.014       0.139
Male SHAP:          -0.112      -0.012      -0.116
Female SHAP:        +0.182      +0.019      +0.184
Counterfactual gap: +6.18pp     +2.68pp     +3.89pp

RF (bagging) reduced the sex bias by 57% through interaction effects.
XGB (boosting) RE-AMPLIFIED it back toward LR levels.

Interpretation: XGBoost's iterative residual-correction process
identified the sex feature as residual-predictive and boosted its
contribution. The most accurate model is not the fairest.

ADDITIONAL FINDING — Credit Amount Surge
-----------------------------------------
LR SHAP: 0.050 | RF SHAP: 0.024 | XGB SHAP: 0.420
An 8× increase over RF. XGBoost found a strong non-linear credit
amount signal that both LR and RF missed. This explains XGBoost's
superior precision — it better distinguishes high-value defaulters.

Dependencies
------------
pip install xgboost scikit-learn imbalanced-learn shap pandas numpy matplotlib
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import shap
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TEST_SIZE    = 0.2

# ── 1. Rebuild model ──────────────────────────────────────────────────────────

df = pd.read_csv("german_credit_cleaned.csv", index_col=0)
X  = df.drop("Risk", axis=1)
y  = df["Risk"]
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

sm           = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = sm.fit_resample(X_train_sc, y_train)
scale_pos_weight = (y_res==0).sum()/(y_res==1).sum()

xgb = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1
)
xgb.fit(X_res, y_res)
y_proba = xgb.predict_proba(X_test_sc)[:, 1]
print("Model retrained.")

# ── 2. SHAP TreeExplainer ─────────────────────────────────────────────────────

explainer = shap.TreeExplainer(xgb)
sv        = explainer.shap_values(X_test_sc)
base_val  = float(explainer.expected_value)
print(f"Base value: {base_val:.4f}")

# ── 3. Global importance ──────────────────────────────────────────────────────

mean_abs = np.abs(sv).mean(axis=0)
imp_df   = pd.DataFrame({
    "feature"      : feature_names,
    "mean_abs_shap": mean_abs
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

print("\n── Global mean |SHAP| ──────────────────────────────────────────────────")
print(imp_df.to_string(index=False))

# ── 4. Sex_male fairness analysis ─────────────────────────────────────────────

sex_idx   = feature_names.index("Sex_male")
male_mask = X_test["Sex_male"].values == 1

sex_rank = imp_df[imp_df["feature"]=="Sex_male"].index[0] + 1
print(f"\n── Sex_male SHAP (fairness) ─────────────────────────────────────────────")
print(f"Rank: #{sex_rank}  Mean |SHAP|: {mean_abs[sex_idx]:.4f}")
print(f"Male SHAP  : {sv[male_mask, sex_idx].mean():.4f}  (LR: -0.112 | RF: -0.012)")
print(f"Female SHAP: {sv[~male_mask, sex_idx].mean():.4f}   (LR: +0.182 | RF: +0.019)")

X_cf_male   = X_test_sc.copy(); X_cf_male[:,sex_idx]   = X_test_sc[:,sex_idx].max()
X_cf_female = X_test_sc.copy(); X_cf_female[:,sex_idx] = X_test_sc[:,sex_idx].min()
gap = xgb.predict_proba(X_cf_female)[:,1].mean() - xgb.predict_proba(X_cf_male)[:,1].mean()
print(f"Counterfactual gap: {gap:+.4f} ({gap*100:.2f}pp)")
print(f"  LR: +6.18pp → RF: +2.68pp → XGB: +{gap*100:.2f}pp")
print(f"  FINDING: Boosting re-amplified the sex bias that bagging reduced.")

# ── 5. Beeswarm ───────────────────────────────────────────────────────────────

shap.summary_plot(sv, X_test_sc, feature_names=feature_names, show=False)
plt.title("SHAP Beeswarm — XGB (German Credit)")
plt.tight_layout()
plt.savefig("shap_beeswarm_xgb_german.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nSaved -> shap_beeswarm_xgb_german.png")

# ── 6. Individual waterfalls ──────────────────────────────────────────────────

sorted_idx = np.argsort(y_proba)
high_idx   = sorted_idx[-5]
low_idx    = sorted_idx[4]
mid_idx    = int(np.argmin(np.abs(y_proba - 0.5)))

for label, idx in [("high_risk", high_idx), ("low_risk", low_idx), ("boundary", mid_idx)]:
    sex = "male" if X_test["Sex_male"].iloc[idx] == 1 else "female"
    print(f"\n── {label} (prob={y_proba[idx]:.4f}, sex={sex}) ──────────────────────")
    row_df = pd.DataFrame({
        "feature": feature_names, "shap": sv[idx]
    }).sort_values("shap", key=abs, ascending=False)
    print(row_df.to_string(index=False))

    shap.waterfall_plot(
        shap.Explanation(
            values=sv[idx], base_values=base_val,
            data=X_test_sc[idx], feature_names=feature_names
        ), show=False
    )
    plt.title(f"SHAP Waterfall — {label} XGB German Credit (sex={sex})")
    plt.tight_layout()
    plt.savefig(f"shap_waterfall_{label}_xgb_german.png", dpi=150, bbox_inches="tight")
    plt.close()

# ── 7. Three-way comparison ────────────────────────────────────────────────────

lr_shap = {"Age":0.133,"Job":0.008,"Housing":0.096,"Saving accounts":0.052,
           "Checking account":0.503,"Credit amount":0.050,"Duration":0.494,
           "Sex_male":0.133,"Purpose_car":0.148,"Purpose_domestic appliances":0.008,
           "Purpose_education":0.120,"Purpose_furniture/equipment":0.144,
           "Purpose_radio/TV":0.081,"Purpose_repairs":0.050,"Purpose_vacation/others":0.069}
rf_shap = {"Checking account":0.137,"Duration":0.079,"Saving accounts":0.044,
           "Credit amount":0.024,"Housing":0.023,"Age":0.017,
           "Purpose_radio/TV":0.016,"Sex_male":0.014,"Purpose_education":0.006,
           "Job":0.006,"Purpose_car":0.006,"Purpose_furniture/equipment":0.002,
           "Purpose_repairs":0.000,"Purpose_domestic appliances":0.000,
           "Purpose_vacation/others":0.000}

comp_df = imp_df.copy()
comp_df["lr_shap"]  = comp_df["feature"].map(lr_shap)
comp_df["rf_shap"]  = comp_df["feature"].map(rf_shap)
comp_df["lr_rank"]  = comp_df["lr_shap"].rank(ascending=False).astype(int)
comp_df["rf_rank"]  = comp_df["rf_shap"].rank(ascending=False).astype(int)
comp_df["xgb_rank"] = comp_df["mean_abs_shap"].rank(ascending=False).astype(int)

print("\n── Three-way SHAP rank comparison ─────────────────────────────────────")
print(comp_df[["feature","lr_rank","rf_rank","xgb_rank",
               "lr_shap","rf_shap","mean_abs_shap"]].to_string(index=False))

# ── 8. Interactions ───────────────────────────────────────────────────────────

sv_df = pd.DataFrame(sv, columns=feature_names)
corr  = sv_df.corr()
pairs = [(feature_names[i], feature_names[j], corr.iloc[i,j])
         for i in range(len(feature_names))
         for j in range(i+1, len(feature_names))
         if not np.isnan(corr.iloc[i,j])]
pairs.sort(key=lambda x: abs(x[2]), reverse=True)
print("\n── Top SHAP correlations ────────────────────────────────────────────────")
for a, b, c in pairs[:5]:
    print(f"  {a[:35]} x {b[:35]}  r={c:.3f}")

print(
    "\nThesis summary — accuracy-fairness paradox:\n"
    "  Most accurate model (XGB, AUC=0.773) is also most biased of\n"
    "  tree models (counterfactual gap=3.89pp vs RF's 2.68pp).\n"
    "  Boosting re-amplifies the sex signal that bagging suppressed.\n"
    "  Credit amount surge (LR=0.050 → XGB=0.420) explains XGB's\n"
    "  precision advantage — it detects high-value defaulters better.\n"
    "  SHAP is the only method that makes these tradeoffs quantifiable."
)
print("\nAll XGB SHAP outputs saved. Proceed to 12_lime_xgb_german.py")

Model retrained.
Base value: -0.0017

── Global mean |SHAP| ──────────────────────────────────────────────────
                    feature  mean_abs_shap
           Checking account       0.837238
                   Duration       0.645346
              Credit amount       0.420128
            Saving accounts       0.352419
                        Age       0.233195
                    Housing       0.177784
                   Sex_male       0.139098
                        Job       0.116271
           Purpose_radio/TV       0.095899
                Purpose_car       0.092154
          Purpose_education       0.079001
Purpose_furniture/equipment       0.016712
            Purpose_repairs       0.009546
Purpose_domestic appliances       0.000000
    Purpose_vacation/others       0.000000

── Sex_male SHAP (fairness) ─────────────────────────────────────────────
Rank: #7  Mean |SHAP|: 0.1391
Male SHAP  : -0.1158  (LR: -0.112 | RF: -0.012)
Female SHAP: 0.1836   (LR: +0.182 | RF: +0.019)


In [ ]:
"""
12_lime_xgb_german.py
======================
LIME explanation of XGBoost on German Credit.
Thesis: Evaluating the Trade-off Between Predictive Accuracy and
        Interpretability in Credit Scoring (XAI Comparative Study)

KEY FINDINGS
------------
R² summary:
  High-risk : 0.095  (unreliable — LIME fails for high-risk XGB)
  Low-risk  : 0.457  (moderate — lower than RF's 0.834)
  Boundary  : 0.426  (moderate — lower than RF's 0.841)

XGB on German Credit is harder for LIME than RF on German Credit,
but easier than XGB on Give Me Some Credit (R²≈0.06 there).
The smaller, simpler German Credit dataset gives LIME more to work with.

Sex_male: LIME rank #15 (last), SHAP rank #7. 8-rank divergence.
LIME would lead a compliance officer to conclude sex is irrelevant.
SHAP shows a 3.89pp counterfactual gap — 3rd worst of all models.

COMPLETE R² PICTURE (all models, both datasets):
  LR  GmSC:    high=0.514  low=0.419  boundary=0.409  mean=0.447
  RF  GmSC:    high=0.198  low=0.549  boundary=0.073  mean=0.273
  XGB GmSC:    high=0.067  low=0.057  boundary=0.056  mean=0.060
  LR  German:  high=0.673  low=0.520  boundary=0.531  mean=0.575
  RF  German:  high=0.215  low=0.834  boundary=0.841  mean=0.630
  XGB German:  high=0.095  low=0.457  boundary=0.426  mean=0.326

Dependencies
------------
pip install xgboost scikit-learn imbalanced-learn lime pandas numpy matplotlib
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from lime.lime_tabular import LimeTabularExplainer
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE  = 42
TEST_SIZE     = 0.2
N_LIME_LOCAL  = 5000
N_LIME_GLOBAL = 150
N_LIME_AGG    = 2000

# ── 1. Rebuild model ──────────────────────────────────────────────────────────

df = pd.read_csv("german_credit_cleaned.csv", index_col=0)
X  = df.drop("Risk", axis=1)
y  = df["Risk"]
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

sm           = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = sm.fit_resample(X_train_sc, y_train)
scale_pos_weight = (y_res==0).sum()/(y_res==1).sum()

xgb = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1
)
xgb.fit(X_res, y_res)
y_proba = xgb.predict_proba(X_test_sc)[:, 1]
print("Model retrained.")

# ── 2. LIME explainer ─────────────────────────────────────────────────────────

explainer = LimeTabularExplainer(
    X_res, feature_names=feature_names,
    class_names=["Good","Bad"], mode="classification",
    random_state=RANDOM_STATE
)

# ── 3. Individual explanations ────────────────────────────────────────────────

sorted_idx = np.argsort(y_proba)
high_idx   = sorted_idx[-5]
low_idx    = sorted_idx[4]
mid_idx    = int(np.argmin(np.abs(y_proba - 0.5)))

def explain_and_plot(idx, label, filename):
    exp = explainer.explain_instance(
        X_test_sc[idx], xgb.predict_proba,
        num_features=10, num_samples=N_LIME_LOCAL
    )
    use_label = 1 if 1 in exp.local_exp else list(exp.local_exp.keys())[-1]
    lime_list = exp.as_list(label=use_label)
    r2        = exp.score
    sex       = "male" if X_test["Sex_male"].iloc[idx] == 1 else "female"

    print(f"\n── {label} (prob={y_proba[idx]:.4f}, R²={r2:.4f}, sex={sex}) ──────────")
    for feat_str, weight in lime_list:
        flag = "  <-- SEX" if "Sex_male" in feat_str else ""
        print(f"  {feat_str:<55} {weight:+.4f}{flag}")
    if r2 < 0.15:
        print(f"  ⚠ R²={r2:.3f} — explanation unreliable")

    return r2

high_r2 = explain_and_plot(high_idx, "HIGH-RISK", "lime_high_risk_xgb_german.png")
low_r2  = explain_and_plot(low_idx,  "LOW-RISK",  "lime_low_risk_xgb_german.png")
mid_r2  = explain_and_plot(mid_idx,  "BOUNDARY",  "lime_boundary_xgb_german.png")

# ── 4. Global aggregation ─────────────────────────────────────────────────────

print(f"\n── Aggregating LIME over {N_LIME_GLOBAL} samples ──────────────────────────────")
np.random.seed(RANDOM_STATE)
sample_idx = np.random.choice(len(X_test_sc), N_LIME_GLOBAL, replace=False)

agg = {f: [] for f in feature_names}
sex_male_w, sex_female_w = [], []

for count, i in enumerate(sample_idx):
    if count % 30 == 0:
        print(f"  {count}/{N_LIME_GLOBAL}...")
    exp = explainer.explain_instance(
        X_test_sc[i], xgb.predict_proba,
        num_features=10, num_samples=N_LIME_AGG
    )
    use_label = 1 if 1 in exp.local_exp else list(exp.local_exp.keys())[-1]
    for feat_str, weight in exp.as_list(label=use_label):
        for fname in feature_names:
            if fname in feat_str:
                agg[fname].append(abs(weight))
                if fname == "Sex_male":
                    if X_test["Sex_male"].iloc[i] == 1:
                        sex_male_w.append(weight)
                    else:
                        sex_female_w.append(weight)
                break

agg_df = pd.DataFrame([
    {"feature": k, "mean_abs_weight": np.mean(v) if v else 0.0}
    for k, v in agg.items()
]).sort_values("mean_abs_weight", ascending=False).reset_index(drop=True)

print("\n── Global LIME importance ──────────────────────────────────────────────")
print(agg_df.to_string(index=False))

print(f"\nSex LIME: male={np.mean(sex_male_w):.4f}  female={np.mean(sex_female_w):.4f}")
print(f"R² summary: high={high_r2:.4f}  low={low_r2:.4f}  boundary={mid_r2:.4f}")

# ── 5. Complete R² comparison across all models ───────────────────────────────

r2_data = {
    "Model":["LR","RF","XGB","LR","RF","XGB"],
    "Dataset":["GmSC","GmSC","GmSC","German","German","German"],
    "High":[0.514,0.198,0.067,0.673,0.215,high_r2],
    "Low" :[0.419,0.549,0.057,0.520,0.834,low_r2],
    "Boundary":[0.409,0.073,0.056,0.531,0.841,mid_r2],
}
r2_df = pd.DataFrame(r2_data)
r2_df["Mean"] = r2_df[["High","Low","Boundary"]].mean(axis=1)

print("\n── Complete LIME R² comparison ─────────────────────────────────────────")
print(r2_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(6)
w = 0.25
labels = [f"{r['Model']}\n{r['Dataset'][:6]}" for _, r in r2_df.iterrows()]
for i, col in enumerate(["High","Low","Boundary"]):
    ax.bar(x + (i-1)*w, r2_df[col], w, label=col, alpha=0.85)
ax.axhline(0.5, color="#d85a30", lw=1.5, linestyle="--", label="R²=0.5 threshold")
ax.axhline(0.1, color="#888", lw=1, linestyle=":", label="R²=0.1 (unreliable)")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("LIME Surrogate R²"); ax.set_ylim(0, 1)
ax.set_title("Complete LIME R² — All Models & Datasets\nKey finding: LR is always most LIME-faithful")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("lime_r2_complete_all_models.png", dpi=150)
plt.close()
print("\nSaved -> lime_r2_complete_all_models.png")

# ── 6. Thesis conclusion ──────────────────────────────────────────────────────

print(
    "\n── FINAL LIME THESIS CONCLUSION ────────────────────────────────────────\n"
    "Sex_male: LIME rank #15 (last) vs SHAP rank #7.\n"
    "LIME would lead a compliance officer to declare sex irrelevant.\n"
    "SHAP shows a 3.89pp counterfactual gap — illegal discrimination.\n"
    "\nRecommendation: SHAP is mandatory for fairness audits.\n"
    "LIME is a useful complement for explaining safe (approved) decisions.\n"
    "Never use LIME alone for regulatory compliance on complex models.\n"
    "\nAll XGB LIME outputs saved. Proceed to 13_pdp_xgb_german.py"
)

Model retrained.

── HIGH-RISK (prob=0.9141, R²=0.0946, sex=male) ──────────
  Duration > 0.53                                         +0.0671
  Purpose_education <= -0.24                              -0.0283
  -1.04 < Checking account <= 0.00                        +0.0254
  Age > 0.48                                              -0.0249
  Job > 0.15                                              -0.0187
  Purpose_radio/TV <= -0.61                               +0.0163
  Saving accounts <= -0.22                                +0.0151
  Credit amount > 0.36                                    +0.0143
  Purpose_vacation/others > -0.09                         +0.0119
  Purpose_repairs <= -0.15                                +0.0074
  ⚠ R²=0.095 — explanation unreliable

── LOW-RISK (prob=0.0095, R²=0.4568, sex=male) ──────────
  Duration <= -0.74                                       -0.1748
  Checking account <= -1.04                               -0.0567
  -0.31 < Credit amount <= 0.36   

In [ ]:
"""
13_pdp_xgb_german.py
=====================
PDP and ICE analysis for XGBoost on German Credit.
Thesis: Evaluating the Trade-off Between Predictive Accuracy and
        Interpretability in Credit Scoring (XAI Comparative Study)

KEY FINDINGS
------------
1. SEX PDP — largest range of all three models:
   LR:  6.27pp (straight line, parallel ICE)
   RF:  4.46pp (shallower, slightly non-parallel)
   XGB: 6.74pp (steeper than RF, non-monotone wobble)
   Three XAI methods (SHAP, LIME, PDP) all confirm: XGBoost
   re-amplified the sex bias that RF reduced.

2. CHECKING ACCOUNT — sharpest threshold:
   "None" → "little": 0.26 → 0.74 (48pp jump).
   Widest PDP range of all features (0.505).
   ICE std≈0.25 — strong interactions in XGB.

3. DURATION — most complex ICE:
   std rises to 0.34 — highest of any model on any feature.
   Non-monotone PDP: dip at mid-range then re-rise.
   Boosting fit local residuals creating irregular shape.

4. CREDIT AMOUNT — deepest U-shape:
   Minimum at mid-range (0.344 at std≈0.18).
   Range 0.265 — much larger than RF (0.078) and LR (0.037).
   Consistent with XGB's 8× SHAP amplification of credit amount.

PDP ranges LR / RF / XGB:
  Checking account: 0.348 / 0.374 / 0.505
  Duration        : 0.429 / 0.261 / 0.411
  Saving accounts : 0.051 / 0.154 / 0.347
  Credit amount   : 0.037 / 0.078 / 0.265
  Sex_male        : 0.063 / 0.045 / 0.067

Dependencies
------------
pip install xgboost scikit-learn imbalanced-learn pandas numpy matplotlib
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import partial_dependence
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE  = 42
TEST_SIZE     = 0.2
N_ICE_SAMPLES = 200
GRID_RES      = 15
ICE_GRID_RES  = 12

# ── 1. Rebuild all three models (for sex PDP comparison) ─────────────────────

df = pd.read_csv("german_credit_cleaned.csv", index_col=0)
X  = df.drop("Risk", axis=1)
y  = df["Risk"]
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

sm           = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = sm.fit_resample(X_train_sc, y_train)
spw          = (y_res==0).sum()/(y_res==1).sum()

lr  = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
rf  = RandomForestClassifier(n_estimators=100, max_depth=10,
      min_samples_leaf=10, class_weight='balanced',
      random_state=RANDOM_STATE, n_jobs=-1)
xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
      subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
      eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)

for m in [lr, rf, xgb]:
    m.fit(X_res, y_res)
print("All three models trained.")

# ── 2. PDP all features (XGB) ────────────────────────────────────────────────

pdp_results = {}
for i, fname in enumerate(feature_names):
    result = partial_dependence(
        xgb, X_res, features=[i],
        kind="average", grid_resolution=GRID_RES, percentiles=(0.05, 0.95)
    )
    pdp_results[fname] = {
        "grid": result["grid_values"][0],
        "avg" : result["average"][0]
    }

range_df = pd.DataFrame([
    {"feature": f, "xgb_range": pdp_results[f]["avg"].max()-pdp_results[f]["avg"].min()}
    for f in feature_names
]).sort_values("xgb_range", ascending=False).reset_index(drop=True)

lr_ranges  = {"Age":0.099,"Job":0.007,"Housing":0.075,"Saving accounts":0.051,
              "Checking account":0.348,"Credit amount":0.037,"Duration":0.429,
              "Sex_male":0.063}
rf_ranges  = {"Age":0.051,"Job":0.046,"Housing":0.065,"Saving accounts":0.154,
              "Checking account":0.374,"Credit amount":0.078,"Duration":0.261,
              "Sex_male":0.045}
range_df["lr_range"] = range_df["feature"].map(lr_ranges).fillna(0)
range_df["rf_range"] = range_df["feature"].map(rf_ranges).fillna(0)

print("\n── PDP ranges: LR / RF / XGB ───────────────────────────────────────────")
print(range_df[["feature","lr_range","rf_range","xgb_range"]].to_string(index=False))

# ── 3. All PDPs plot ──────────────────────────────────────────────────────────

fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()
for i, fname in enumerate(feature_names):
    ax = axes[i]
    c  = "#d85a30" if fname == "Sex_male" else "#b07d2a"
    ax.plot(pdp_results[fname]["grid"], pdp_results[fname]["avg"], color=c, lw=2)
    ax.set_title(fname[:20]+(" ⚠" if fname=="Sex_male" else ""), fontsize=9,
                 color="#b83820" if fname=="Sex_male" else "black")
    ax.tick_params(labelsize=7)
    ax.set_ylabel("Pred. prob." if i%5==0 else "")

for j in range(len(feature_names), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("PDP — XGB (German Credit)  |  Red=Sex_male fairness flag",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig("pdp_all_xgb_german.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nSaved -> pdp_all_xgb_german.png")

# ── 4. PDP + ICE focus features ──────────────────────────────────────────────

np.random.seed(RANDOM_STATE)
ice_sample = X_res[np.random.choice(len(X_res), N_ICE_SAMPLES, replace=False)]

focus = ["Checking account", "Duration", "Saving accounts", "Sex_male"]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, fname in zip(axes, focus):
    fi    = feature_names.index(fname)
    color = "#d85a30" if fname == "Sex_male" else "#b07d2a"

    pdp_r = partial_dependence(xgb, ice_sample, features=[fi],
                               kind="average", grid_resolution=GRID_RES,
                               percentiles=(0.05, 0.95))
    ice_r = partial_dependence(xgb, ice_sample, features=[fi],
                               kind="individual", grid_resolution=ICE_GRID_RES,
                               percentiles=(0.05, 0.95))

    grid_ice = ice_r["grid_values"][0]
    ice_vals = ice_r["individual"][0]
    mean_ice = ice_vals.mean(axis=0)
    std_ice  = ice_vals.std(axis=0)

    for line in ice_vals[::5]:
        ax.plot(grid_ice, line, color="#d85a30", alpha=0.05, lw=0.7)
    ax.fill_between(grid_ice, mean_ice-std_ice, mean_ice+std_ice,
                    color="#d85a30", alpha=0.15, label="ICE ±1 SD")
    ax.plot(pdp_r["grid_values"][0], pdp_r["average"][0],
            color=color, lw=2.5, label="PDP", zorder=5)

    title = fname + (" ⚠ FAIRNESS" if fname == "Sex_male" else "")
    ax.set_title(title, fontsize=10,
                 color="#b83820" if fname == "Sex_male" else "black")
    ax.set_xlabel(fname[:22], fontsize=9)
    ax.set_ylabel("Pred. default prob." if fname == focus[0] else "")
    ax.legend(fontsize=8)

plt.suptitle("PDP + ICE — XGB (German Credit)\nHighest ICE fan-out of all models",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig("pdp_ice_xgb_german.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved -> pdp_ice_xgb_german.png")

# ── 5. Sex PDP three-model comparison ────────────────────────────────────────

sex_fi = feature_names.index("Sex_male")
pdp_lr  = partial_dependence(lr,  X_res, features=[sex_fi], kind="average",
                              grid_resolution=20, percentiles=(0.05, 0.95))
pdp_rf  = partial_dependence(rf,  X_res, features=[sex_fi], kind="average",
                              grid_resolution=20, percentiles=(0.05, 0.95))
pdp_xgb = partial_dependence(xgb, X_res, features=[sex_fi], kind="average",
                              grid_resolution=20, percentiles=(0.05, 0.95))

lr_range  = float(pdp_lr["average"][0].max()  - pdp_lr["average"][0].min())
rf_range  = float(pdp_rf["average"][0].max()  - pdp_rf["average"][0].min())
xgb_range = float(pdp_xgb["average"][0].max() - pdp_xgb["average"][0].min())

print(f"\n── Sex PDP ranges ──────────────────────────────────────────────────────")
print(f"LR:  {lr_range:.4f} ({lr_range*100:.2f}pp)")
print(f"RF:  {rf_range:.4f} ({rf_range*100:.2f}pp)")
print(f"XGB: {xgb_range:.4f} ({xgb_range*100:.2f}pp)  ← largest")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pdp_lr["grid_values"][0],  pdp_lr["average"][0],
        color="#3266ad", lw=2, linestyle="--", label=f"LR ({lr_range*100:.1f}pp)")
ax.plot(pdp_rf["grid_values"][0],  pdp_rf["average"][0],
        color="#2a7a3b", lw=2, linestyle=":", label=f"RF ({rf_range*100:.1f}pp)")
ax.plot(pdp_xgb["grid_values"][0], pdp_xgb["average"][0],
        color="#b07d2a", lw=2.5, label=f"XGB ({xgb_range*100:.1f}pp)")
ax.set_xlabel("Sex_male (standardised)")
ax.set_ylabel("Predicted default probability")
ax.set_title("Sex PDP — LR vs RF vs XGB\nXGB re-amplifies sex bias that RF reduced")
ax.legend()
plt.tight_layout()
plt.savefig("pdp_sex_all_models_german.png", dpi=150)
plt.close()
print("Saved -> pdp_sex_all_models_german.png")

print(
    "\nGerman Credit analysis COMPLETE — all scripts 01–13 saved.\n"
    "Three models × three XAI methods × PDP+ICE all done.\n"
    "\nKey German Credit findings:\n"
    "  1. RF/XGB give genuine +11pp AUC over LR (unlike GmSC)\n"
    "  2. Sex bias trajectory: LR 6.18pp → RF 2.68pp → XGB 3.89pp\n"
    "  3. XGB re-amplifies bias RF reduced — accuracy-fairness paradox\n"
    "  4. SHAP is the only tool that captures the full fairness picture\n"
    "  5. LIME misses sex discrimination in all complex models\n"
    "  6. PDP confirms fairness finding across all three models visually\n"
    "\nReady for thesis synthesis Chapter 4."
)

All three models trained.


ValueError: cannot reshape array of size 1 into shape (2)